In [ ]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
# Folder containing the CSV files
DATA_DIR = Path(r"../datasets")
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)




# Load ratings.csv
ratings_path = DATA_DIR / "raw/ratings.csv"

if not ratings_path.exists():
    raise FileNotFoundError(f"ratings.csv not found: {ratings_path}")

ratings = pd.read_csv(ratings_path)

print("Shape:", ratings.shape)
print("Columns:", ratings.columns.tolist())
display(ratings.head())



In [ ]:
# 4. LOAD MOVIES FOR LATER ANALYSIS
movies_path = DATA_DIR / "raw/movies.csv"

if movies_path.exists():
    movies_lookup = pd.read_csv(movies_path)
    print("Movies:", movies_lookup.shape)
else:
    movies_lookup = None
    print("movies.csv not found; rating processing can continue.")

In [ ]:
#  REQUIRED COLUMN VALIDATION
required_columns = ["userId", "movieId", "rating"]
missing_columns = [c for c in required_columns if c not in ratings.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
print("Required columns are present.")

In [ ]:
#DATA TYPES AND MISSING VALUES
print(ratings.dtypes)
print("\nMissing values:")
display(ratings.isna().sum().rename("missing_count").to_frame())

In [ ]:
# NORMALIZE CORE TYPES
ratings["userId"] = pd.to_numeric(ratings["userId"], errors="coerce")
ratings["movieId"] = pd.to_numeric(ratings["movieId"], errors="coerce")
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")

if "timestamp" in ratings.columns:
    ratings["timestamp"] = pd.to_numeric(ratings["timestamp"], errors="coerce")

ratings = ratings.dropna(subset=["userId", "movieId", "rating"]).copy()

ratings["userId"] = ratings["userId"].astype("int64")
ratings["movieId"] = ratings["movieId"].astype("int64")
ratings["rating"] = ratings["rating"].astype("float32")

print(ratings.dtypes)

In [ ]:
# DUPLICATE USER/MOVIE CHECK
duplicate_mask = ratings.duplicated(["userId", "movieId"], keep=False)

print("Rows in duplicate user/movie groups:", duplicate_mask.sum())

if duplicate_mask.any():
    display(
        ratings[duplicate_mask]
        .sort_values(["userId", "movieId"])
        .head(20)
    )


In [ ]:
#  CREATE CLEAN RATINGS
if "timestamp" in ratings.columns:
    ratings_clean = (
        ratings.groupby(["userId", "movieId"], as_index=False)
        .agg(
            rating=("rating", "mean"),
            timestamp=("timestamp", "max")
        )
    )
else:
    ratings_clean = (
        ratings.groupby(["userId", "movieId"], as_index=False)
        .agg(rating=("rating", "mean"))
    )

ratings_clean["rating"] = ratings_clean["rating"].round(2).astype("float32")

print("Raw rows:", len(ratings))
print("Clean rows:", len(ratings_clean))
display(ratings_clean.head())


In [ ]:
# RATING DISTRIBUTION
rating_counts = ratings_clean["rating"].value_counts().sort_index()
display(rating_counts.rename("count").to_frame())

print("\nStatistics:")
display(ratings_clean["rating"].describe())


In [ ]:
# RATING CHART
rating_counts.plot(kind="bar", figsize=(9, 5))
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# DATASET SIZE
total_ratings = len(ratings_clean)
unique_users = ratings_clean["userId"].nunique()
unique_movies = ratings_clean["movieId"].nunique()

print("Total ratings:", total_ratings)
print("Unique users:", unique_users)
print("Unique movies:", unique_movies)


In [ ]:
# RATINGS PER USER
ratings_per_user = ratings_clean.groupby("userId").size().rename("rating_count")
display(ratings_per_user.describe())

print("Most active users:")
display(ratings_per_user.sort_values(ascending=False).head(20).to_frame())


In [ ]:
# USER ACTIVITY GROUPS
user_activity = ratings_per_user.to_frame()

user_activity["activity_group"] = pd.cut(
    user_activity["rating_count"],
    bins=[0, 5, 10, 20, 50, 100, np.inf],
    labels=["1-5", "6-10", "11-20", "21-50", "51-100", "100+"]
)

display(
    user_activity["activity_group"]
    .value_counts()
    .sort_index()
    .rename("users")
    .to_frame()
)


In [ ]:
# RATINGS PER MOVIE
ratings_per_movie = ratings_clean.groupby("movieId").size().rename("rating_count")
display(ratings_per_movie.describe())

print("Most rated movies:")
display(ratings_per_movie.sort_values(ascending=False).head(20).to_frame())


In [ ]:
# MOVIE ACTIVITY GROUPS
movie_activity = ratings_per_movie.to_frame()

movie_activity["activity_group"] = pd.cut(
    movie_activity["rating_count"],
    bins=[0, 5, 10, 20, 50, 100, np.inf],
    labels=["1-5", "6-10", "11-20", "21-50", "51-100", "100+"]
)

display(
    movie_activity["activity_group"]
    .value_counts()
    .sort_index()
    .rename("movies")
    .to_frame()
)


In [ ]:
#  DATASET SPARSITY
possible_pairs = unique_users * unique_movies
density = total_ratings / possible_pairs
sparsity = 1 - density

print(f"Possible user/movie pairs: {possible_pairs:,}")
print(f"Observed ratings: {total_ratings:,}")
print(f"Density: {density:.6%}")
print(f"Sparsity: {sparsity:.6%}")


In [ ]:
#  USER RATING STATISTICS
user_rating_stats = (
    ratings_clean.groupby("userId")["rating"]
    .agg(
        rating_count="count",
        average_rating="mean",
        rating_std="std",
        min_rating="min",
        max_rating="max"
    )
    .reset_index()
)

display(user_rating_stats.head(10))


In [ ]:
# MOVIE RATING STATISTICS
movie_rating_stats = (
    ratings_clean.groupby("movieId")["rating"]
    .agg(
        rating_count="count",
        average_rating="mean",
        rating_std="std",
        min_rating="min",
        max_rating="max"
    )
    .reset_index()
)

if movies_lookup is not None and {"movieId", "title"}.issubset(movies_lookup.columns):
    movie_rating_stats = movie_rating_stats.merge(
        movies_lookup[["movieId", "title"]].drop_duplicates("movieId"),
        on="movieId",
        how="left"
    )

display(movie_rating_stats.head(10))


In [ ]:
# TOP-RATED MOVIES WITH MINIMUM SUPPORT
MIN_MOVIE_RATINGS = 100

top_rated = (
    movie_rating_stats[
        movie_rating_stats["rating_count"] >= MIN_MOVIE_RATINGS
    ]
    .sort_values(
        ["average_rating", "rating_count"],
        ascending=[False, False]
    )
    .head(20)
)

display(top_rated)


In [ ]:
# ELIGIBLE USERS
MIN_USER_RATINGS = 5

eligible_users = ratings_per_user[
    ratings_per_user >= MIN_USER_RATINGS
].index

print("Total users:", unique_users)
print("Eligible users:", len(eligible_users))
print("Excluded users:", unique_users - len(eligible_users))

ratings_model = ratings_clean[
    ratings_clean["userId"].isin(eligible_users)
].copy()

print("Modeling rows:", len(ratings_model))


In [19]:
# USER-LEVEL TRAIN/TEST SPLIT
RANDOM_STATE = 42
TEST_SIZE = 0.20

def split_user_ratings(group):
    if len(group) < 2:
        return group.copy(), group.iloc[0:0].copy()

    test_count = max(1, int(np.ceil(len(group) * TEST_SIZE)))

    # Always keep at least one training rating.
    test_count = min(test_count, len(group) - 1)

    test_part = group.sample(
        n=test_count,
        random_state=RANDOM_STATE
    )

    train_part = group.drop(test_part.index)

    return train_part, test_part


train_parts = []
test_parts = []

for user_id, group in ratings_model.groupby("userId"):
    train_part, test_part = split_user_ratings(group)
    train_parts.append(train_part)
    test_parts.append(test_part)

train_ratings = pd.concat(train_parts, ignore_index=True)
test_ratings = pd.concat(test_parts, ignore_index=True)

print("Training rows:", len(train_ratings))
print("Test rows:", len(test_ratings))


Training rows: 19936012
Test rows: 5064083


In [ ]:
# TRAIN/TEST VALIDATION
print("Training users:", train_ratings["userId"].nunique())
print("Test users:", test_ratings["userId"].nunique())

users_without_training = (
    set(test_ratings["userId"])
    - set(train_ratings["userId"])
)

print("Test users without training history:", len(users_without_training))

assert len(users_without_training) == 0


In [ ]:
# DATA LEAKAGE CHECK
train_pairs = set(
    zip(train_ratings["userId"], train_ratings["movieId"])
)

test_pairs = set(
    zip(test_ratings["userId"], test_ratings["movieId"])
)

overlap_pairs = train_pairs.intersection(test_pairs)

print("Train/test user-movie overlap:", len(overlap_pairs))

assert len(overlap_pairs) == 0

print("DATA LEAKAGE CHECK PASSED")


In [ ]:
# POSITIVE RATINGS
POSITIVE_RATING_THRESHOLD = 4.0

positive_train_ratings = train_ratings[
    train_ratings["rating"] >= POSITIVE_RATING_THRESHOLD
].copy()

positive_test_ratings = test_ratings[
    test_ratings["rating"] >= POSITIVE_RATING_THRESHOLD
].copy()

print("Positive train ratings:", len(positive_train_ratings))
print("Positive test ratings:", len(positive_test_ratings))


In [ ]:
# USER PREFERENCE STATISTICS
user_preference_stats = (
    train_ratings.groupby("userId")["rating"]
    .agg(
        rating_count="count",
        average_rating="mean"
    )
    .reset_index()
)

positive_counts = (
    positive_train_ratings.groupby("userId")
    .size()
    .rename("positive_rating_count")
    .reset_index()
)

user_preference_stats = user_preference_stats.merge(
    positive_counts,
    on="userId",
    how="left"
)

user_preference_stats["positive_rating_count"] = (
    user_preference_stats["positive_rating_count"]
    .fillna(0)
    .astype(int)
)

display(user_preference_stats.head(10))


In [ ]:
# 28. EXPORT PROCESSED DATASETS
outputs = {
    "ratings_clean.csv": ratings_clean,
    "ratings_train.csv": train_ratings,
    "ratings_test.csv": test_ratings,
    "ratings_positive_train.csv": positive_train_ratings,
    "ratings_positive_test.csv": positive_test_ratings,
    "user_rating_stats.csv": user_rating_stats,
    "movie_rating_stats.csv": movie_rating_stats,
    "user_preference_stats.csv": user_preference_stats,
}

for filename, dataframe in outputs.items():
    output_path = OUTPUT_DIR / filename
    dataframe.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")


In [ ]:
#FINAL VALIDATION
assert len(ratings_clean) > 0
assert ratings_clean["userId"].notna().all()
assert ratings_clean["movieId"].notna().all()
assert ratings_clean["rating"].notna().all()
assert ratings_clean["rating"].between(0, 5).all()

assert not ratings_clean.duplicated(
    subset=["userId", "movieId"]
).any()

assert len(overlap_pairs) == 0

assert set(test_ratings["userId"]).issubset(
    set(train_ratings["userId"])
)

print("=" * 60)
print("PHASE 3 RATING PROCESSING PASSED")
print("=" * 60)
print("Clean ratings:", len(ratings_clean))
print("Users:", ratings_clean["userId"].nunique())
print("Movies:", ratings_clean["movieId"].nunique())
print("Train ratings:", len(train_ratings))
print("Test ratings:", len(test_ratings))
print("Positive train ratings:", len(positive_train_ratings))
print("Positive test ratings:", len(positive_test_ratings))
